# Bài 7 · Chơi Tic-Tac-Toe tương tác

Đối thủ là `and_or_player` ở Bài 2. `interactive_player` in bàn cờ trước lượt của bạn, dùng chỉ số `0–8` cho ô còn trống. Nhập chỉ số của một ô trống để đi hoặc `-1` để dừng ván. Hai bản ghi dưới đây dùng chuỗi nhập đã định sẵn, đi qua cùng hàm đọc và kiểm tra nước đi như khi chơi trực tiếp.

In [1]:
from and_or_tictactoe import and_or_player
from tictactoe import actions, check_board, empty_board, other, result, terminal


class GameStopped(Exception):
    """The person playing asked to end the game."""


def show_grid(board):
    grid = [mark.upper() if mark != ' ' else str(i) for i, mark in enumerate(board)]
    print('\n---------\n'.join(' | '.join(grid[i:i + 3]) for i in (0, 3, 6)))


def interactive_player(board, player=None, *, input_fn=input):
    print(f"\nLượt của bạn ({player.upper() if player else '?'})")
    show_grid(board)
    legal = actions(board)
    while True:
        try:
            move = int(input_fn('Chọn ô hoặc nhập -1 để dừng: '))
        except ValueError:
            print('Hãy nhập một số nguyên.')
            continue
        if move == -1:
            raise GameStopped
        if move in legal:
            return move
        print(f'Ô này không hợp lệ. Các ô còn trống: {legal}')


def play_interactive(human='x', *, input_fn=input, opponent=and_or_player):
    if human not in {'x', 'o'}:
        raise ValueError('Vai chơi phải là x hoặc o.')
    board, player = empty_board(), 'x'
    history = []
    while not terminal(board):
        if player == human:
            try:
                move = interactive_player(board, player, input_fn=input_fn)
            except GameStopped:
                print('Đã dừng ván.')
                return history, None
        else:
            move = opponent(board, player)
            print(f'Tác tử ({player.upper()}) đi ô {move}.')
        board = result(board, player, move)
        history.append((player, move))
        player = other(player)
    print('\nBàn cờ cuối ván')
    show_grid(board)
    outcome = check_board(board)
    print('Hòa.' if outcome == 'd' else f'{outcome.upper()} thắng.')
    return history, outcome

## Hai ván thử

Lần thử vai X cố ý nhập `abc` để kiểm tra xử lý lỗi rồi tiếp tục đánh. Lần thử vai O cố ý nhập vào ô 0 đã có quân. Mỗi ván lưu lượt và ô đã đi để có thể dựng lại bàn cờ cuối.

In [2]:
def scripted_input(moves):
    script = iter(moves)

    def read(prompt):
        entry = next(script)
        print(prompt + str(entry))
        return str(entry)

    return read


print('Ván 1 · Người chơi vai X')
history_x, outcome_x = play_interactive('x', input_fn=scripted_input(['abc', 4, 8, 1, 3, 5, 6, 7, 2]))
print('Lịch sử:', history_x)
print('\nVán 2 · Người chơi vai O')
history_o, outcome_o = play_interactive('o', input_fn=scripted_input([0, 4, 1, 5, 6, 7, 8, 2, 3]))
print('Lịch sử:', history_o)

for history, outcome in [(history_x, outcome_x), (history_o, outcome_o)]:
    reconstructed = empty_board()
    for player, move in history:
        assert not terminal(reconstructed) and move in actions(reconstructed)
        reconstructed = result(reconstructed, player, move)
    assert terminal(reconstructed) and check_board(reconstructed) == outcome

Ván 1 · Người chơi vai X

Lượt của bạn (X)
0 | 1 | 2
---------
3 | 4 | 5
---------
6 | 7 | 8
Chọn ô hoặc nhập -1 để dừng: abc
Hãy nhập một số nguyên.
Chọn ô hoặc nhập -1 để dừng: 4
Tác tử (O) đi ô 0.

Lượt của bạn (X)
O | 1 | 2
---------
3 | X | 5
---------
6 | 7 | 8
Chọn ô hoặc nhập -1 để dừng: 8
Tác tử (O) đi ô 2.

Lượt của bạn (X)
O | 1 | O
---------
3 | X | 5
---------
6 | 7 | X
Chọn ô hoặc nhập -1 để dừng: 1
Tác tử (O) đi ô 7.

Lượt của bạn (X)
O | X | O
---------
3 | X | 5
---------
6 | O | X
Chọn ô hoặc nhập -1 để dừng: 3
Tác tử (O) đi ô 5.

Lượt của bạn (X)
O | X | O
---------
X | X | O
---------
6 | O | X
Chọn ô hoặc nhập -1 để dừng: 5
Ô này không hợp lệ. Các ô còn trống: [6]
Chọn ô hoặc nhập -1 để dừng: 6

Bàn cờ cuối ván
O | X | O
---------
X | X | O
---------
X | O | X
Hòa.
Lịch sử: [('x', 4), ('o', 0), ('x', 8), ('o', 2), ('x', 1), ('o', 7), ('x', 3), ('o', 5), ('x', 6)]

Ván 2 · Người chơi vai O
Tác tử (X) đi ô 0.

Lượt của bạn (O)
X | 1 | 2
---------
3 | 4 | 5
---------


Khi người chơi cầm X, tác tử O chọn ô 7 để chặn cột `1–4–7`, rồi chọn ô 5 để chặn hàng `3–4–5`. Ván kết thúc hòa sau chín nước. Khi người chơi cầm O, sau các nước `X0, O4, X1`, O chọn ô 5 và bỏ trống ô 2. Tác tử X đi ô 2 để hoàn thành hàng `0–1–2` và thắng sau năm nước.

## Chơi trực tiếp

Đổi `PLAY_AS` thành `"x"` hoặc `"o"` trong ô dưới rồi chạy lại ô đó. Giá trị mặc định cho phép chạy cả notebook liên tục mà không cần nhập.

In [3]:
PLAY_AS = None
if PLAY_AS in {'x', 'o'}:
    play_interactive(PLAY_AS)